In [1]:
import pandas as pd

In [2]:
data = {
    'user_id': [1, 1, 1, 2, 2, 2],
    'timestamp': ['10:00', '10:01', '10:02', '10:00', '10:01', '10:02'],
    'ping_ms': [20, 22, 21, 30, 32, 150] # User 2 spikes at 10:02
}
df = pd.DataFrame(data)
df

,user_id,timestamp,ping_ms
0,1,10:00,20
1,1,10:01,22
2,1,10:02,21
3,2,10:00,30
4,2,10:01,32
5,2,10:02,150


In [3]:
type(df)

pandas.DataFrame

In [4]:
df = df.sort_values(by=['user_id', 'timestamp'])

In [5]:
df

,user_id,timestamp,ping_ms
0,1,10:00,20
1,1,10:01,22
2,1,10:02,21
3,2,10:00,30
4,2,10:01,32
5,2,10:02,150


In [6]:
df['ping_spike'] = df.groupby('user_id')['ping_ms'].diff()

In [7]:
# df.drop(columns=['ping_difference'], axis = 0, inplace=True)
# df

In [8]:
df

,user_id,timestamp,ping_ms,ping_spike
0,1,10:00,20,NaN
1,1,10:01,22,2.0
2,1,10:02,21,-1.0
3,2,10:00,30,NaN
4,2,10:01,32,2.0
5,2,10:02,150,118.0


In [9]:
anomalies = df[df['ping_spike'] > 50]
anomalies

,user_id,timestamp,ping_ms,ping_spike
5,2,10:02,150,118.0


In [10]:
# df['regional_rank'] = df.groupby('region')['session_duration_minutes'].rank(method='dense', ascending=False)

In [11]:
import pandas as pd

data = {
    'session_id': [101, 102, 103, 104, 105, 106, 107],
    'region': ['US-West', 'US-West', 'US-West', 'EU-Central', 'EU-Central', 'US-East', 'US-East'],
    'user_id': [1, 2, 3, 4, 5, 6, 7],
    'duration_minutes': [120, 120, 45, 90, 85, 200, 15] #tie at 120 in US-West
}
df = pd.DataFrame(data)

# 2. Apply DENSE_RANK partitioned by region (groupby) and ordered descending
df['regional_rank'] = df.groupby('region')['duration_minutes'].rank(method='dense', ascending=False)

# 3. Filter for only the top 2 longest sessions per region
top_sessions = df[df['regional_rank'] <= 2].sort_values(by=['region', 'regional_rank'])

print("Full DataFrame with Ranks:")
print(df.sort_values(by=['region', 'regional_rank']))
print("\n---")
print("Top 2 Sessions Per Region:")
print(top_sessions)

Full DataFrame with Ranks:
   session_id      region  user_id  duration_minutes  regional_rank
3         104  EU-Central        4                90            1.0
4         105  EU-Central        5                85            2.0
5         106     US-East        6               200            1.0
6         107     US-East        7                15            2.0
0         101     US-West        1               120            1.0
1         102     US-West        2               120            1.0
2         103     US-West        3                45            2.0

---
Top 2 Sessions Per Region:
   session_id      region  user_id  duration_minutes  regional_rank
3         104  EU-Central        4                90            1.0
4         105  EU-Central        5                85            2.0
5         106     US-East        6               200            1.0
6         107     US-East        7                15            2.0
0         101     US-West        1               120     

In [1]:
import pandas as pd
import numpy as np

print("--- 1. CREATING THE TELEMETRY DATA ---")
# We intentionally inject NaN (missing data) to practice cleaning
data = {
    'user_id': [1, 1, 1, 2, 2, 3, 3, 3],
    'timestamp': pd.to_datetime([
        '2026-03-02 20:00:00', '2026-03-02 20:05:00', '2026-03-02 20:10:00',
        '2026-03-02 20:00:00', '2026-03-02 20:15:00', 
        '2026-03-02 20:00:00', '2026-03-02 20:05:00', '2026-03-02 20:10:00'
    ]),
    'ping_ms': [20.0, np.nan, 25.0, 30.0, 300.0, 15.0, 16.0, np.nan],
    'game_fps': [60, 59, 60, 45, 12, 120, 118, 119],
    'region': ['US-West', 'US-West', 'US-West', 'EU-Central', 'EU-Central', 'US-East', np.nan, 'US-East']
}
df = pd.DataFrame(data)
print(df.head())

print("\n--- 2. THE BASICS: HANDLING NULLS (NaN) ---")
# Strategy A: Fill missing ping values with the median ping of the dataset
median_ping = df['ping_ms'].median()
df['ping_ms'] = df['ping_ms'].fillna(median_ping)

# Strategy B: Drop rows where the 'region' is completely missing (corrupted data)
df = df.dropna(subset=['region'])
print("After fillna (ping) and dropna (region):")
print(df)

print("\n--- 3. FILTERING & SORTING ---")
# Find all sessions where FPS dropped below 60 AND ping was over 50
bad_sessions = df[(df['game_fps'] < 60) & (df['ping_ms'] > 50)]
print("Bad Sessions:")
print(bad_sessions)

# Always sort time-series data before doing advanced math!
df = df.sort_values(by=['user_id', 'timestamp'])

print("\n--- 4. INTERMEDIATE: GROUPBY & AGGREGATION ---")
# Get the max ping and average FPS per user
user_stats = df.groupby('user_id').agg(
    max_ping=('ping_ms', 'max'),
    avg_fps=('game_fps', 'mean'),
    session_count=('timestamp', 'count')
).reset_index() # reset_index() flattens the dataframe so it's easy to read
print(user_stats)

print("\n--- 5. ADVANCED: TIME-SERIES WINDOW FUNCTIONS ---")
# 5A. LAG/LEAD (Shift): What was this user's ping 5 minutes ago?
df['previous_ping'] = df.groupby('user_id')['ping_ms'].shift(1)

# 5B. DIFF: How much did the ping spike from the last reading?
df['ping_spike_size'] = df.groupby('user_id')['ping_ms'].diff()

# 5C. ROLLING WINDOW: 2-period moving average of FPS per user
# Notice the min_periods=1 ensures it doesn't output NaN on the first row
df['moving_avg_fps'] = df.groupby('user_id')['game_fps'].transform(lambda x: x.rolling(window=2, min_periods=1).mean())

print(df[['user_id', 'timestamp', 'ping_ms', 'ping_spike_size', 'game_fps', 'moving_avg_fps']])

print("\n--- 6. ADVANCED: RANKING ---")
# Rank users in each region by their highest ping (1 = worst connection)
df['regional_ping_rank'] = df.groupby('region')['ping_ms'].rank(method='dense', ascending=False)
print(df[['user_id', 'region', 'ping_ms', 'regional_ping_rank']].sort_values(by=['region', 'regional_ping_rank']))

print("\n--- 7. ADVANCED: MERGING (SQL JOINS) ---")
# Merging a secondary table (like a SQL LEFT JOIN)
server_data = pd.DataFrame({
    'region': ['US-West', 'EU-Central', 'US-East', 'AP-South'],
    'server_status': ['Online', 'Degraded', 'Online', 'Offline']
})
# Merge on the 'region' column
final_df = pd.merge(df, server_data, on='region', how='left')
print(final_df[['user_id', 'region', 'server_status']].head())

--- 1. CREATING THE TELEMETRY DATA ---
   user_id           timestamp  ping_ms  game_fps      region
0        1 2026-03-02 20:00:00     20.0        60     US-West
1        1 2026-03-02 20:05:00      NaN        59     US-West
2        1 2026-03-02 20:10:00     25.0        60     US-West
3        2 2026-03-02 20:00:00     30.0        45  EU-Central
4        2 2026-03-02 20:15:00    300.0        12  EU-Central

--- 2. THE BASICS: HANDLING NULLS (NaN) ---
After fillna (ping) and dropna (region):
   user_id           timestamp  ping_ms  game_fps      region
0        1 2026-03-02 20:00:00     20.0        60     US-West
1        1 2026-03-02 20:05:00     22.5        59     US-West
2        1 2026-03-02 20:10:00     25.0        60     US-West
3        2 2026-03-02 20:00:00     30.0        45  EU-Central
4        2 2026-03-02 20:15:00    300.0        12  EU-Central
5        3 2026-03-02 20:00:00     15.0       120     US-East
7        3 2026-03-02 20:10:00     22.5       119     US-East

--- 3.